# DSN Mart — Standalone 1067.13140 Pipeline

This notebook reconstructs the recovered end-to-end pipeline using only:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

It does **not** require pre-generated prediction files.

Recovered final recipe:

**50% Audited GBR + 25% StoreCategoryLinear (min 30) + 25% ShrunkStoreCategory (alpha 100)**

The Audited GBR is rebuilt from the recovered `advanced_solution.py` + `audit_1087.py` logic.
The store-price baseline and store-category components are rebuilt directly from the raw training data.


In [ ]:
# Optional install cell for a fresh environment.
# Uncomment if needed:
# !pip install -q pandas numpy scikit-learn

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

SEED = 2026
TARGET = "total_sales"
ROOT = Path(".")

TRAIN_PATH = ROOT / "train.csv"
TEST_PATH = ROOT / "test.csv"
SAMPLE_PATH = ROOT / "sample_submission.csv"

OUTPUT_PATH = ROOT / "submission_beat1068_robust.csv"

BASE_PARAMS = dict(
    n_estimators=600,
    learning_rate=0.025,
    max_depth=2,
    min_samples_leaf=10,
    min_samples_split=12,
    loss="squared_error",
    subsample=1.0,
    random_state=SEED,
)


## 1. Load and validate raw competition files


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

assert TARGET in train_raw.columns
assert TARGET not in test_raw.columns
assert "id" in train_raw and "id" in test_raw and "id" in sample
assert train_raw["id"].is_unique
assert test_raw["id"].is_unique
assert sample["id"].is_unique
assert len(test_raw) == len(sample)
assert set(test_raw["id"]) == set(sample["id"])

print("Train:", train_raw.shape)
print("Test :", test_raw.shape)
print("Sample:", sample.shape)
train_raw.head()


## 2. Recovered cleaning logic


In [ ]:
def clean(df):
    z = df.copy()

    z["product_weight_kg"] = pd.to_numeric(
        z["product_weight_kg"], errors="coerce"
    )

    categorical = [
        "product_code", "fat_content", "product_category",
        "store_code", "store_size", "store_location_tier",
        "store_format"
    ]

    for c in categorical:
        z[c] = z[c].astype("string").str.strip()

    z["product_category"] = (
        z["product_category"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .fillna("missing")
    )

    z["fat_content"] = (
        z["fat_content"]
        .str.lower()
        .replace({
            "lf": "low fat",
            "low-fat": "low fat",
            "reg": "regular"
        })
        .fillna("missing")
    )

    for c in [
        "product_code", "store_code", "store_size",
        "store_location_tier", "store_format"
    ]:
        z[c] = z[c].fillna("missing")

    z["category_store"] = z["product_category"] + "__" + z["store_code"]
    z["category_location"] = z["product_category"] + "__" + z["store_location_tier"]
    z["category_format"] = z["product_category"] + "__" + z["store_format"]
    z["fat_store"] = z["fat_content"] + "__" + z["store_code"]
    z["product_format"] = z["product_code"] + "__" + z["store_format"]
    z["product_location"] = z["product_code"] + "__" + z["store_location_tier"]

    return z

train = clean(train_raw)
test = clean(test_raw)

y = train[TARGET].reset_index(drop=True)
x = train.drop(columns=TARGET).reset_index(drop=True)
test_x = test.reset_index(drop=True)

print("Cleaned train:", train.shape)


## 3. Exact target-free deterministic features

These are predictor-only transformations. No `total_sales` aggregation enters the Audited GBR.


In [ ]:
def deterministic_features(source_x, target_x):
    z = target_x.copy()

    product_weight = source_x.groupby("product_code")["product_weight_kg"].median()

    z["weight_missing"] = z["product_weight_kg"].isna().astype(int)
    z["store_size_missing"] = (z["store_size"] == "missing").astype(int)

    z["product_weight_kg"] = (
        z["product_weight_kg"]
        .fillna(z["product_code"].map(product_weight))
        .fillna(source_x["product_weight_kg"].median())
    )

    z["log_price"] = np.log1p(z["product_price"])
    z["price_sq"] = z["product_price"] ** 2

    z["price_per_kg"] = (
        z["product_price"] /
        z["product_weight_kg"].replace(0, np.nan)
    )
    z["price_per_kg"] = z["price_per_kg"].fillna(
        z["price_per_kg"].median()
    )

    z["log_visibility"] = np.log1p(z["shelf_visibility"])
    z["visibility_zero"] = (z["shelf_visibility"] == 0).astype(int)
    z["visibility_sq"] = z["shelf_visibility"] ** 2

    cat_price = source_x.groupby("product_category")["product_price"].agg(
        ["median", "mean"]
    )
    store_price = source_x.groupby("store_code")["product_price"].mean()
    product_price = source_x.groupby("product_code")["product_price"].mean()
    cat_vis = source_x.groupby("product_category")["shelf_visibility"].mean()
    store_vis = source_x.groupby("store_code")["shelf_visibility"].mean()

    global_price_median = source_x["product_price"].median()
    global_price_mean = source_x["product_price"].mean()

    z["price_minus_category_median"] = (
        z["product_price"] -
        z["product_category"].map(cat_price["median"]).fillna(global_price_median)
    )

    z["price_to_category_mean"] = (
        z["product_price"] /
        z["product_category"].map(cat_price["mean"]).fillna(global_price_mean)
    )

    z["price_minus_product_mean"] = (
        z["product_price"] -
        z["product_code"].map(product_price).fillna(global_price_mean)
    )

    z["price_minus_store_mean"] = (
        z["product_price"] -
        z["store_code"].map(store_price).fillna(global_price_mean)
    )

    z["visibility_minus_category_mean"] = (
        z["shelf_visibility"] -
        z["product_category"].map(cat_vis).fillna(source_x["shelf_visibility"].mean())
    )

    z["visibility_minus_store_mean"] = (
        z["shelf_visibility"] -
        z["store_code"].map(store_vis).fillna(source_x["shelf_visibility"].mean())
    )

    z["price_x_visibility"] = z["product_price"] * z["shelf_visibility"]
    z["price_x_age"] = z["product_price"] * z["store_age_years"]
    z["age_sq"] = z["store_age_years"] ** 2

    for key in [
        "product_code", "store_code",
        "product_category", "category_store"
    ]:
        z[f"{key}_frequency"] = (
            z[key]
            .map(source_x[key].value_counts())
            .fillna(0)
            .astype(float)
        )

    return z


## 4. Exact Audited GBR feature selection


In [ ]:
def strict_base_features(source_x, target_x):
    z = deterministic_features(source_x, target_x)

    numeric = z.select_dtypes(include=[np.number]).columns.tolist()

    removed = {
        "log_price",
        "price_sq",
        "log_visibility",
        "visibility_sq",
        "visibility_zero",
        "price_x_visibility",
        "price_x_age",
        "weight_missing",
        "store_size_missing",
        "price_to_category_mean",
        "price_minus_product_mean",
        "price_minus_store_mean",
        "visibility_minus_category_mean",
        "visibility_minus_store_mean",
    }

    keep = [c for c in numeric if c not in removed]

    return (
        z[keep]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0),
        keep,
    )

full_base, base_columns = strict_base_features(x, x)
test_base, test_columns = strict_base_features(x, test_x)

assert base_columns == test_columns
assert list(full_base.columns) == list(test_base.columns)

print("Audited GBR feature count:", len(base_columns))
print(base_columns)


## 5. Train the Audited GBR from scratch

Recovered configuration:

- GradientBoostingRegressor
- 600 estimators
- learning rate 0.025
- max depth 2
- minimum leaf size 10
- minimum split size 12
- squared-error loss
- seed 2026


In [ ]:
audited_gbr_model = GradientBoostingRegressor(**BASE_PARAMS)
audited_gbr_model.fit(full_base, y)

audited_gbr = audited_gbr_model.predict(test_base)
audited_gbr = np.maximum(audited_gbr, 0)

assert len(audited_gbr) == len(test_raw)
assert np.isfinite(audited_gbr).all()

print("Audited GBR:")
print(pd.Series(audited_gbr).describe())


## 6. Rebuild the selected Store-price Linear model

Recovered design:

- standardized price: `(price - 140) / 60`
- store one-hot indicators
- store × standardized-price interactions
- ordinary LinearRegression


In [ ]:
def price_design(rows, degree=1):
    p = rows["product_price"].astype(float).to_numpy()
    ps = (p - 140.0) / 60.0

    store = pd.get_dummies(
        rows["store_code"].astype("string"),
        prefix="store",
        dtype=float
    ).reset_index(drop=True)

    z = pd.DataFrame({"price_z": ps})

    for d in range(2, degree + 1):
        z[f"price_z_{d}"] = ps ** d

    z = pd.concat([z, store], axis=1)

    for d in range(1, degree + 1):
        q = store.mul(ps ** d, axis=0)
        q.columns = [f"store_price{d}__{c}" for c in q.columns]
        z = pd.concat([z, q], axis=1)

    return z

def align_columns(train_features, target_features):
    return target_features.reindex(
        columns=train_features.columns,
        fill_value=0.0
    )

price_train = price_design(x, degree=1)
price_test = price_design(test_x, degree=1)
price_test = align_columns(price_train, price_test)

store_price_model = LinearRegression()
store_price_model.fit(price_train, y)

store_price = store_price_model.predict(price_test)
store_price = np.maximum(store_price, 0)

assert np.isfinite(store_price).all()

print("Store-price Linear:")
print(pd.Series(store_price).describe())


## 7. Store × category local linear prediction

For every `(store_code, product_category)`:

- fit a local linear relationship between standardized product price and sales;
- require at least 5 observations and at least 3 unique prices before fitting;
- otherwise retain the Store-price Linear fallback.


In [ ]:
def store_category_linear_predictions(
    train_frame,
    test_frame,
    baseline
):
    result = np.asarray(baseline, dtype=float).copy()
    counts = np.zeros(len(test_frame), dtype=int)

    for (store, category), fit in train_frame.groupby(
        ["store_code", "product_category"],
        observed=True
    ):
        mask = (
            test_frame["store_code"].eq(store) &
            test_frame["product_category"].eq(category)
        ).to_numpy()

        if not mask.any():
            continue

        n = len(fit)
        counts[mask] = n

        if n < 5 or fit["product_price"].nunique() < 3:
            continue

        mu = float(fit["product_price"].mean())
        sd = max(
            float(fit["product_price"].std(ddof=0)),
            1e-6
        )

        x_fit = (
            (fit["product_price"].to_numpy(float) - mu) / sd
        ).reshape(-1, 1)

        x_test_local = (
            (test_frame.loc[mask, "product_price"].to_numpy(float) - mu) / sd
        ).reshape(-1, 1)

        model = LinearRegression()
        model.fit(x_fit, fit[TARGET].to_numpy(float))

        result[mask] = model.predict(x_test_local)

    return np.maximum(result, 0), counts

category_linear, category_count = store_category_linear_predictions(
    train,
    test,
    store_price
)

print("Rows with >= 5 historical observations:",
      int((category_count >= 5).sum()))

print("Rows with >= 30 historical observations:",
      int((category_count >= 30).sum()))


## 8. Build the two recovered store-category components


In [ ]:
# Component C1:
# use local store-category linear model only for groups with n >= 30.
store_category_hard30 = np.where(
    category_count >= 30,
    category_linear,
    store_price
)

# Component C2:
# local model shrunk toward store-price baseline.
# weight = n / (n + 100)
shrink_weight = np.where(
    category_count >= 5,
    category_count / (category_count + 100.0),
    0.0
)

store_category_shrink100 = (
    shrink_weight * category_linear +
    (1.0 - shrink_weight) * store_price
)

store_category_hard30 = np.maximum(
    store_category_hard30, 0
)
store_category_shrink100 = np.maximum(
    store_category_shrink100, 0
)

print("Hard30 mean :", store_category_hard30.mean())
print("Shrink100 mean:", store_category_shrink100.mean())


## 9. Final recovered 50 / 25 / 25 blend

Final prediction:

`0.50 * AuditedGBR + 0.25 * StoreCategoryHard30 + 0.25 * StoreCategoryShrink100`


In [ ]:
final_prediction = (
    0.50 * audited_gbr +
    0.25 * store_category_hard30 +
    0.25 * store_category_shrink100
)

final_prediction = np.maximum(final_prediction, 0)

assert len(final_prediction) == len(test_raw)
assert np.isfinite(final_prediction).all()
assert (final_prediction >= 0).all()

print(pd.Series(final_prediction).describe())


## 10. Create submission in the exact sample-submission ID order


In [ ]:
prediction_map = pd.DataFrame({
    "id": test_raw["id"],
    TARGET: final_prediction
})

submission = (
    sample[["id"]]
    .merge(
        prediction_map,
        on="id",
        how="left",
        validate="one_to_one"
    )
)

assert list(submission.columns) == ["id", TARGET]
assert len(submission) == len(test_raw)
assert submission["id"].equals(sample["id"])
assert submission[TARGET].notna().all()
assert np.isfinite(submission[TARGET]).all()

submission.to_csv(OUTPUT_PATH, index=False)

print(f"Saved successfully: {OUTPUT_PATH}")
print("Shape:", submission.shape)
submission.head(10)


## 11. Reproducibility summary

This notebook now trains all required prediction components itself.

It does **not** load:

- `submission_gbr_1087_audited.csv`
- `submission_masked_optimized.csv`
- `submission_blend_400.csv`
- any `.pkl` / `.joblib` model
- any OOF prediction file

Required competition inputs are only:

1. `train.csv`
2. `test.csv`
3. `sample_submission.csv`

The generated file is:

**`submission_beat1068_robust.csv`**

The historical **1067.13140 leaderboard score** is external leaderboard evidence. Running this notebook reproduces the recovered modeling recipe; the notebook itself cannot independently recompute a private leaderboard score.
